# 实验 4：二维 pullback 方向到底来自哪里？

实验 3 已经证明：在**已知真实边界位置**时，$G=J^	op J$ 的最大特征方向很贴近边界法向。但原来的 overlap 与 energy 在它们各自估计的曲线上读取，比较混入了位置误差。

本实验把所有观测量放到同一个真实边界点，问题只有一个：

> 有限时间 $G$ 是否包含同点 overlap、energy 和局部线性稳定性没有的方向信息？

四个控制方向是：

1. $e_{max}(G_t)$；
2. $
abla_{u,v}[m_A(x_t)-m_B(x_t)]$；
3. 参数空间 energy Hessian 的最负曲率方向；
4. $S_0=J_0^	op(Df_0^	op+Df_0)J_0$ 与逐时刻 $S_t=J_t^	op(Df_t^	op+Df_t)J_t$ 的最大特征方向。

$S_0$ 是冻结的初始局部控制；$S_t$ 使用了当前轨迹与切映射，只作为逐时刻机制诊断，二者不混用。


## 1. 下载并核对冻结源码

Notebook 固定到已经运行正式实验并通过测试的提交。任何源码哈希不一致都会立即停止。


In [ ]:
import hashlib
import importlib.util
from pathlib import Path
import subprocess
import sys
import urllib.request

required = {
    "jax": "jax[cpu]",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

CODE_REV = "e959678527d55cc3f93cba9ae8dd8e9e5d84cebb"
BASE = Path("/content/hopfield-dynamic-geometry")
SOURCES = {
    "experiment_01_boundary_localization.py": "c40bf773180ccd7ebced4c9413851878ae868e27b0c7207b3d1ff1abd0f79ec2",
    "experiment_02_moving_boundary.py": "58bd4699e94bfc11b83678b1b305280008269d42977b2d67ecfc41c5420da86b",
    "experiment_03_2d_directional_geometry.py": "353d9d931b24539646e2a585f79f516d6efa1753150c18146bf93d660401139e",
    "experiment_04_direction_mechanism_controls.py": "b641ea39013a221c12095daca884cbdf8ccb9e68251c5ec417314f944fde025c",
}
BASE.mkdir(parents=True, exist_ok=True)
raw_root = f"https://raw.githubusercontent.com/Heptazero/nn-labs/{CODE_REV}/representation-geometry/experiments/hopfield-dynamic-geometry"
for name, expected in SOURCES.items():
    target = BASE / name
    urllib.request.urlretrieve(f"{raw_root}/{name}", target)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"SHA-256 mismatch for {name}: {actual}")
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
print("verified source revision:", CODE_REV)


## 2. 冻结条件与停止门

保持实验 3 的网络、查询、5 个 $delta$、8 个 seed、时间网格和真实边界完全不变。只新增同位置方向控制。

主要统计仍以 seed 为独立单位做配对 bootstrap，实用裕量冻结为 $1^circ$。

- 若 $G$ 超过同点 overlap、energy 和 $S_0$：进入全平面盲定位。
- 若 $G$ 超过 overlap、energy，但与 $S_0$ 相同：机制降级为局部线性稳定性。
- 若同点 overlap 或 energy 追平 $G$：实验 3 的独立优势不成立，保留 $G$ 为几何表示。


In [ ]:
from experiment_04_direction_mechanism_controls import (
    MechanismControlConfig,
    run_experiment,
    write_artifacts,
)

config = MechanismControlConfig()
output_dir = Path("/content/experiment_04")
conditions, boundaries, directions, raw = run_experiment(config)
summary, conclusion_path = write_artifacts(
    output_dir, conditions, boundaries, directions, raw, config
)
summary


## 3. 查看公平的同位置结果


In [ ]:
from IPython.display import display, Image

display(Image(filename=str(output_dir / "main_figure.png")))
display(
    directions[
        directions["time"].isin([config.early_mechanism_time, 1.0, 6.0])
    ].groupby("time")[
        [
            "G_angle_deg",
            "overlap_angle_deg",
            "energy_angle_deg",
            "strain_S0_angle_deg",
            "strain_St_angle_deg",
            "G_vs_S0_angle_deg",
            "G_linearization_relative_error",
        ]
    ].median()
)
print(conclusion_path.read_text())


## 4. 已冻结正式运行的判读

仓库中随源码保存的 5 deltas × 8 seeds 正式运行得到：

- 40/40 条件保持有效拓扑；
- $tge1$ 的中位法向误差：$G=0.109^circ$，同点 overlap $=0.109^circ$，同点 energy $=0.109^circ$，$S_t=0.109^circ$；
- 三个同点动态量相对 $G$ 的配对差异都在 $10^{-5}$ 度以内，没有达到 $1^circ$ 实用裕量；
- 固定 $S_0$ 的晚期误差约 $4.00^circ$，但在 $t=0.02$ 时，$G$ 与 $S_0$ 方向只差 $0.104^circ$，且 $G_0+tS_0$ 的相对误差为 $3.31%$。

因此结果属于第三条停止规则：同位置控制消除了 $G$ 对 overlap 与 energy 的优势。实验 3 仍证明了 $G$ 能描述边界方向，但没有证明它在这个双记忆系统中提供独立诊断信息。短时 $G$ 方向则可由局部线性应变解释。


## 5. 下载全部原始产物


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/experiment_04_artifacts", "zip", root_dir=output_dir
)
print("archive:", archive)
files.download(archive)
